# Homework 9 — Working with molecules (RDKit)

**Due Monday Oct 26.** Open notes, open lecture, open internet.

Thursday's lecture turned text into molecules and read numbers off them. This homework
does the same thing one small step at a time, and finishes on a real dataset of 1,128
compounds whose water solubility was measured in a lab.

**How this works**
1. Run the **Setup** cell. It installs RDKit — about 30 seconds, and nothing below
   works without it.
2. Work through the numbered sections in order. Each one has a worked example you can
   copy from.
3. There are **twenty problems**. Each one tells you exactly what to do in a **Do this**
   list — follow the steps in order and you are done with that problem.
4. The problems are **independent**: every one builds the molecules it needs in its own
   cell and none of them uses a name from an earlier problem. If one gives you trouble,
   skip it — the rest still work.
5. Run the **Check in** cell at the very end and type its number into the CHEM 438 app.

Most problems are two or three lines of code. If one is taking you much longer than
that, re-read the worked example just above it.

<!-- c438-copy-note -->
> ### ▶️ First — make your own copy
> This notebook opened **read-only** (it comes straight from GitHub). To type your answers and keep your work:
>
> **File ▸ Save a copy in Drive**  —  or click **Copy to Drive** in the toolbar.
>
> Then work in *your* copy. It saves to your Google Drive automatically.

In [ ]:
# Setup — run this first
!pip install rdkit -q
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, PandasTools
print("RDKit is ready.")

## 1. From text to a molecule

A **SMILES** is a molecule written as text. `Chem.MolFromSmiles` turns it into a
molecule object you can measure and draw.

In [ ]:
ethanol = Chem.MolFromSmiles("CCO")
print(ethanol.GetNumAtoms())
ethanol

Three atoms, and in Colab the molecule draws itself.

**The one trap:** if the text is not a valid SMILES, RDKit hands back `None` instead of
raising an error. Nothing goes wrong until the next line, where `None` has no
`.GetNumAtoms()`.

In [ ]:
broken = Chem.MolFromSmiles("CQ")     # Q is not an element
print(broken)

**Problem 1 (predict, then run).** Parse one good SMILES and one bad one, and see what
the difference looks like.

**Do this:**
1. Parse `"CCO"` with `Chem.MolFromSmiles` and print the number of atoms.
2. Parse `"CCXO"` — `X` is not an element — and **print the result itself**, not its
   atom count.
3. In a `#` comment, write what RDKit handed back for `"CCXO"`, and why step 2 did not
   raise an error.

In [ ]:
# Problem 1

**Problem 2 (predict, then run).** RDKit counts **heavy** atoms — everything except
hydrogen. Find out how that differs from the count you would write down by hand.

**Do this:**
1. *Before running anything*, write down how many atoms you expect each of these to
   have: methane `C`, water `O`, propane `CCC`, acetic acid `CC(=O)O`.
2. Build all four in the cell below and print each one's `GetNumAtoms()`.
3. In a `#` comment, name the one you got wrong — if any — and say why.

In [ ]:
# Problem 2

**Problem 3 (read the error, fix the bug).** Make the atom-count code survive a bad
SMILES instead of crashing on it.

A classmate writes:

```python
mol = Chem.MolFromSmiles("CCXO")
print(mol.GetNumAtoms())
```

and gets:

```
AttributeError: 'NoneType' object has no attribute 'GetNumAtoms'
```

Write a version that does not crash when the SMILES is bad.

**Do this:**
1. Loop over the two SMILES `"CCXO"` and `"CCO"`.
2. Parse each one. If the result **is** `None`, print the SMILES and `could not parse`;
   otherwise print the SMILES and its atom count.
3. In a `#` comment, say in one sentence what the `AttributeError` was really telling
   them.

In [ ]:
# Problem 3

## 2. Where the hydrogens are

RDKit keeps hydrogens implicit, because for drawing and for most calculations they are
bookkeeping. `Chem.AddHs` makes them real atoms, and `atom.GetTotalNumHs()` asks a
single atom how many it is carrying.

In [ ]:
m = Chem.MolFromSmiles("CCO")
print("heavy atoms: ", m.GetNumAtoms())
print("molar mass:  ", Descriptors.MolWt(m))

**Problem 4 (predict, then run).** `Chem.AddHs(mol)` returns a **new** molecule with the
hydrogens written out. Find out what that does and does not change.

**Do this:**
1. *Before running anything*, write down what you expect `Chem.AddHs(mol).GetNumAtoms()`
   to be for ethanol `CCO`, and whether `Descriptors.MolWt` will change.
2. Build ethanol, call `Chem.AddHs` on it, and print the atom count **before and
   after**.
3. Print the molar mass before and after, each rounded to 3 decimals.
4. In a `#` comment, answer: **did the molar mass change, and why not?**

In [ ]:
# Problem 4

**Problem 5 (loop the atoms).** Work out propan-1-ol's molecular formula from its atoms
instead of looking it up. Propan-1-ol is `CCCO`.

**Do this:**
1. Build the molecule.
2. Loop over its atoms with `for atom in mol.GetAtoms():` and print each atom's
   `GetSymbol()` and its `GetTotalNumHs()`.
3. Print the **total** number of hydrogens across all atoms.
4. In a `#` comment, write the molecular formula those numbers give you.

In [ ]:
# Problem 5

**Problem 6 (interpret the numbers).** Benzene is `c1ccccc1` — six aromatic carbons in a
ring. Work out how many hydrogens each of those carbons is carrying.

**Do this:**
1. Build benzene and print its heavy atom count.
2. Print the atom count again after `Chem.AddHs`.
3. In a `#` comment, say how many hydrogens that means **each carbon** carries, and why
   it is not the three that a carbon in propane carries.

In [ ]:
# Problem 6

## 3. Measuring a molecule

A **descriptor** is one number computed from the structure. Each one takes a molecule
and gives back a number.

In [ ]:
caffeine = Chem.MolFromSmiles("Cn1cnc2c1c(=O)n(C)c(=O)n2C")
print("MW         ", Descriptors.MolWt(caffeine))
print("logP       ", Descriptors.MolLogP(caffeine))
print("H donors   ", Descriptors.NumHDonors(caffeine))
print("H acceptors", Descriptors.NumHAcceptors(caffeine))

**logP** is the one to remember: it is how a molecule splits between oil and water.
**Positive means greasy, negative means water-loving.** Caffeine comes out at −1.03,
which is why it dissolves in your coffee.

**Problem 7 (read the error, fix the bug).** Get ethanol's molar mass, starting from a
line that crashes.

A classmate tries it and writes:

```python
Descriptors.MolWt("CCO")
```

It crashes with:

```
ArgumentError: Python argument types in rdkit.Chem.Descriptors.MolWt(str)
did not match C++ signature
```

**Do this:**
1. Write a version that prints ethanol's molar mass **without** crashing.
2. In a `#` comment, say in one sentence what was wrong with their line.

In [ ]:
# Problem 7

**Problem 8 (measure a molecule).** Get the same four numbers for aspirin that the
worked example above got for caffeine. Aspirin is `CC(=O)Oc1ccccc1C(=O)O`.

**Do this:**
1. Build aspirin.
2. Print its molecular weight, logP, H-bond donors and H-bond acceptors, each on its own
   line with a label.

In [ ]:
# Problem 8

**Problem 9 (predict, then run).** Decide which of two molecules is heavier, then check.
Caffeine is `Cn1cnc2c1c(=O)n(C)c(=O)n2C` and aspirin is `CC(=O)Oc1ccccc1C(=O)O`.

**Do this:**
1. *Before running anything*, guess which one is heavier.
2. Build both and print each molecular weight rounded to 2 decimals.
3. In a `#` comment, say whether your guess was right, and how far apart the two numbers
   actually are.

In [ ]:
# Problem 9

**Problem 10 (interpret the numbers).** Check logP against something you already know.
Ethanol (`CCO`) mixes with water in any amount; hexane (`CCCCCC`) does not mix with
water at all.

**Do this:**
1. Print the logP of each, rounded to 2 decimals.
2. In a `#` comment, say which **sign** goes with "mixes with water".
3. In the same comment, say whether your two numbers agree with what you already know
   about these two liquids, and why.

In [ ]:
# Problem 10

**Problem 11 (interpret the numbers).** Use logP to tell apart two solids you have
handled: glucose, which dissolves in water, and naphthalene — mothballs — which does
not.

- glucose `OCC1OC(O)C(O)C(O)C1O`
- naphthalene `c1ccc2ccccc2c1`

**Do this:**
1. Print the logP of each, rounded to 2 decimals.
2. Print the number of H-bond donors of each (`Descriptors.NumHDonors`).
3. In a `#` comment, say which number — logP or donor count — you would quote to explain
   why sugar dissolves in water and mothballs do not, and what about glucose's structure
   produces it.

In [ ]:
# Problem 11

## 4. A table of molecules

A DataFrame column can hold molecules, and then `apply` runs a descriptor down the whole
column — unit 9's `apply`, with an RDKit function in it.

In [ ]:
df = pd.DataFrame({"name": ["water", "ethanol", "benzene"],
                   "smiles": ["O", "CCO", "c1ccccc1"]})
PandasTools.AddMoleculeColumnToFrame(frame=df, smilesCol="smiles")
df["mw"] = df["ROMol"].apply(Descriptors.MolWt)
df

`AddMoleculeColumnToFrame` adds a column called **`ROMol`** holding the molecules. That
is the column `apply` works on — not the `smiles` column.

**Problem 12 (put it together).** Build your own table of four drugs and rank them by
how greasy they are.

| name | SMILES |
|---|---|
| ethanol | `CCO` |
| caffeine | `Cn1cnc2c1c(=O)n(C)c(=O)n2C` |
| aspirin | `CC(=O)Oc1ccccc1C(=O)O` |
| ibuprofen | `CC(C)Cc1ccc(cc1)C(C)C(=O)O` |

**Do this:**
1. Build a DataFrame with a `name` column and a `smiles` column from the table above.
2. Add the molecule column with `AddMoleculeColumnToFrame`.
3. Add a `logp` column with `apply`.
4. Print `name` and `logp` sorted with the **greasiest first**.
5. In a `#` comment, name the greasiest of the four.

In [ ]:
# Problem 12

**Problem 13 (read the error, fix the bug).** Get a `mw` column onto a DataFrame,
starting from code that raises a `KeyError`.

A classmate writes:

```python
df = pd.DataFrame({"name": ["ethanol"], "smiles": ["CCO"]})
df["mw"] = df["ROMol"].apply(Descriptors.MolWt)
```

and gets:

```
KeyError: 'ROMol'
```

**Do this:**
1. Write the version that works — build the same one-row frame, then add the `mw` column.
2. Print `name` and `mw`.
3. In a `#` comment, say which line they left out, and what a `KeyError` tells you in
   general.

In [ ]:
# Problem 13

**Problem 14 (loop the rows).** Build a table of five common solvents and find the
heaviest one.

| name | SMILES |
|---|---|
| water | `O` |
| ethanol | `CCO` |
| acetone | `CC(=O)C` |
| toluene | `Cc1ccccc1` |
| hexane | `CCCCCC` |

**Do this:**
1. Build a DataFrame with `name` and `smiles` from the table above.
2. Add the molecule column, then a `mw` column **and** a `logp` column.
3. Print `name`, `mw` and `logp` for all five.
4. Print the **name** of the one with the largest `mw`.

*Hint for step 4: `df.loc[df["mw"].idxmax(), "name"]`.*

In [ ]:
# Problem 14

**Problem 15 (interpret the numbers).** You need to dissolve a very greasy compound,
something with a logP around 4. Pick which of those same five solvents to reach for.

**Do this:**
1. Build the five solvents again in the cell below, so this problem stands on its own.
2. Add the molecule column and a `logp` column.
3. Print `name` and `logp` sorted with the greasiest first.
4. In a `#` comment, name the solvent you would use, say which **column** decided it,
   and say why the heaviest solvent is not the answer.

In [ ]:
# Problem 15

## 5. Does it contain...?

A **SMARTS** is a pattern rather than a molecule. `HasSubstructMatch` answers yes or no.

In [ ]:
alcohol = Chem.MolFromSmarts("[CX4][OX2H]")
print(Chem.MolFromSmiles("CCO").HasSubstructMatch(alcohol))
print(Chem.MolFromSmiles("c1ccccc1").HasSubstructMatch(alcohol))

**Problem 16 (does it contain?).** Find out which of four molecules contain a benzene
ring. The pattern is `c1ccccc1` — all lowercase, because those are aromatic carbons.

The four: ethanol `CCO`, caffeine `Cn1cnc2c1c(=O)n(C)c(=O)n2C`, aspirin
`CC(=O)Oc1ccccc1C(=O)O`, ibuprofen `CC(C)Cc1ccc(cc1)C(C)C(=O)O`.

**Do this:**
1. Build the pattern with `Chem.MolFromSmarts`.
2. Loop over the four molecules and print each name with `True` or `False` from
   `HasSubstructMatch`.
3. In a `#` comment, say how many of the four have one.
4. Caffeine's answer may surprise you. In the same comment, look at its rings and say
   why it comes out the way it does.

In [ ]:
# Problem 16

**Problem 17 (does it contain?).** Now find the carboxylic acids — the `-COOH` group.
The pattern is `[CX3](=O)[OX2H1]`.

Test the same four as Problem 16: ethanol `CCO`, caffeine
`Cn1cnc2c1c(=O)n(C)c(=O)n2C`, aspirin `CC(=O)Oc1ccccc1C(=O)O`, ibuprofen
`CC(C)Cc1ccc(cc1)C(C)C(=O)O`.

**Do this:**
1. Build the pattern and print each name with `True` or `False`.
2. In a `#` comment, name the ones that have a `-COOH`.
3. In the same comment, connect that to a number you printed in **Problem 8**: a `-COOH`
   is why aspirin had the H-bond donor count it did.
4. Ethanol has an `-OH` but comes back `False`. Say why that is correct.

In [ ]:
# Problem 17

**Problem 18 (fix the pattern).** Caffeine plainly has two C=O groups — you can see them
in its SMILES, `Cn1cnc2c1c(=O)n(C)c(=O)n2C`. But searching it for `C=O` finds **zero**
of them. Find out why, and write a pattern that works.

**Do this:**
1. Build caffeine.
2. Print how many matches `Chem.MolFromSmarts("C=O")` finds, using
   `len(caffeine.GetSubstructMatches(pattern))`.
3. Print the same count for the pattern `[#6]=[OX1]`, which asks for **any** carbon
   double-bonded to an oxygen.
4. In a `#` comment, explain the difference between the two counts. *The rule you need:
   in SMARTS, case carries meaning — lowercase `c` is an aromatic carbon, uppercase `C`
   is an aliphatic one, and `[#6]` is "carbon, either way." Now look at where caffeine's
   two carbonyl carbons sit.*

In [ ]:
# Problem 18

## 6. A real dataset

Everything above was molecules you typed. **ESOL** (Delaney, 2004) is 1,128 compounds
whose solubility in water was actually measured in a lab. `read_csv` takes a URL, so
there is nothing to upload.

In [ ]:
url = "https://raw.githubusercontent.com/jyesselm/chem438-notebooks/main/data/esol_solubility.csv"
esol = pd.read_csv(url)
print(esol.shape)
esol[["Compound ID", "smiles"]].head()

The measured value lives in a long-named column:
`"measured log solubility in mols per litre"`. It is a **log** solubility, so −1 is very soluble and −9 is essentially
insoluble.

**Problem 19 (read the data).** Get your bearings in the ESOL file before computing
anything from it.

**Do this:**
1. Load the CSV in your own cell from this URL:
   `https://raw.githubusercontent.com/jyesselm/chem438-notebooks/main/data/esol_solubility.csv`
2. Print its shape.
3. Print the smallest and the largest value of the column
   `"measured log solubility in mols per litre"`, each rounded to 2 decimals.
4. Print the `Compound ID` of the **most** soluble compound.
5. In a `#` comment, say how many orders of magnitude the column spans, and why that
   explains it being stored as a log.

*Hint for step 4: `df.loc[df[col].idxmax(), "Compound ID"]`.*

In [ ]:
# Problem 19

**Problem 20 (put it together, real data).** Compute one descriptor across all 1,128
compounds and see what the dataset is made of.

**Do this:**
1. Load the same CSV again in your own cell.
2. Add a molecule column with `AddMoleculeColumnToFrame`.
3. Add a `logp` column with `apply`.
4. Print the **mean** logP across all 1,128 compounds, rounded to 2 decimals.
5. In a `#` comment, say whether the average compound in this file is greasy or
   water-loving, and whether that is what you would expect from a dataset about things
   dissolving.

*This one takes a few seconds to run — 1,128 molecules is a lot more than four.*

In [ ]:
# Problem 20

## Check in

Run the cell below. It reloads the solubility file and counts how many of the 1,128
compounds satisfy **all four** of Lipinski's rules for a drug you can swallow as a
pill — molecular weight ≤ 500, logP ≤ 5, H-bond donors ≤ 5, H-bond acceptors ≤ 10.

It prints one number. **Type that number into the CHEM 438 app to mark this homework
complete.**

In [ ]:
check = pd.read_csv("https://raw.githubusercontent.com/jyesselm/chem438-notebooks/main/data/esol_solubility.csv")
mols = [Chem.MolFromSmiles(s) for s in check["smiles"]]
passes = 0
for mol in mols:
    if (Descriptors.MolWt(mol) <= 500 and Descriptors.MolLogP(mol) <= 5
            and Descriptors.NumHDonors(mol) <= 5
            and Descriptors.NumHAcceptors(mol) <= 10):
        passes = passes + 1
print("Your check-in value:", passes)